# Week 54, Day 6: Forge: Week 54 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w054/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
print("1.", 0.02 + 0.05 + 0.1)
print("2.", round(0.5 + 0.05 * 1.87, 3))
print("3.", 2 ** (1 / 0.25), "times")

### Exercise: Part A: two hyperparameters, chosen honestly

For every degree and penalty, fit ridge (on scaled polynomial features, intercept unpenalised) to the 100 training trials and record the validation MSE. Refit the chosen pair on all 150 trials and evaluate once on the test set. Also compute oracle_err, the lowest test error over the whole grid when refitting on all 150.

In [ ]:
import numpy as np
r = np.random.default_rng(546)
f = lambda s: 3 * np.exp(-((s - 0.3) / 0.35) ** 2) + 1          # a tuning curve
s_all = r.uniform(-1, 1, 150); y_all = f(s_all) + 0.5 * r.normal(size=150)
s_te = r.uniform(-1, 1, 5000); y_te = f(s_te) + 0.5 * r.normal(size=5000)
tr, va = np.arange(100), np.arange(100, 150)

def feats(s, deg):
    return np.vander(s, deg + 1, increasing=True)

def fit(s, y, deg, lam):
    A = feats(s, deg); D = np.eye(deg + 1); D[0, 0] = 0
    return np.linalg.solve(A.T @ A + lam * D, A.T @ y)

mse = lambda s, y, w, deg: np.mean((y - feats(s, deg) @ w) ** 2)
degs, lams = np.arange(1, 11), np.logspace(-6, 1, 15)
val_err = np.zeros((10, len(lams)))
for i, deg in enumerate(degs):
    for j, lam in enumerate(lams):
        val_err[i, j] = ...
i, j = np.unravel_index(np.argmin(val_err), val_err.shape)
best_deg, best_lam = degs[i], lams[j]
honest_err = ...
oracle_err = min(mse(s_te, y_te, fit(s_all, y_all, d, l), d) for d in degs for l in lams)
print(f"chosen: degree {best_deg}, lambda {best_lam:.1e}; validation MSE {val_err.min():.3f}")
print(f"honest test MSE {honest_err:.3f}; best possible on the grid {oracle_err:.3f}; noise floor 0.25")

In [ ]:
import numpy as np
def test_grid():
    "val_err has one entry per degree and penalty"
    assert val_err.shape == (10, len(lams))
def test_choice():
    "the chosen pair minimises the validation error"
    i, j = np.unravel_index(np.argmin(val_err), val_err.shape)
    assert (best_deg, best_lam) == (degs[i], lams[j])
def test_honest():
    "the honest test error is within 15% of the best achievable, and the validation error of the chosen pair is lower than its test error"
    assert honest_err <= 1.15 * oracle_err and val_err.min() < honest_err

test_grid()
test_choice()
test_honest()
print("All checks passed")

### Exercise: Part B: interpretable or accurate?

For each training size, average over 10 training sets the test MSE (on 2,000 test trials) of a linear regression with intercept and of 10-nearest-neighbour regression. Store (linear, knn) in results[n].

In [ ]:
g = np.random.default_rng(547)
truth = lambda X: 0.5 * X[:, 0] * X[:, 1] + X[:, 0]
def draw(n):
    X = g.normal(size=(n, 2)); return X, truth(X) + 0.5 * g.normal(size=n)
X_te, y_te2 = draw(2000)

def knn(X_tr, y_tr, X_q, k=10):
    d = ((X_q[:, None, :] - X_tr[None, :, :]) ** 2).sum(-1)
    return y_tr[np.argsort(d, axis=1)[:, :k]].mean(1)

results = {}
for n in [25, 2000]:
    lin, nn = [], []
    for rep in range(10):
        X_tr, y_tr = draw(n)
        w, *_ = np.linalg.lstsq(np.c_[X_tr, np.ones(n)], y_tr, rcond=None)
        lin.append(...)
        nn.append(...)
    results[n] = (np.mean(lin), np.mean(nn))
    print(f"n = {n:4d}: linear model {results[n][0]:.3f}, 10 nearest neighbours {results[n][1]:.3f}")
print("linear coefficients (last fit):", np.round(w, 2))

In [ ]:
import numpy as np
def test_small():
    "with 25 trials the linear model is more accurate"
    assert results[25][0] < results[25][1]
def test_large():
    "with 2,000 trials the nearest-neighbour model is clearly more accurate"
    assert results[2000][1] < 0.8 * results[2000][0]

test_small()
test_large()
print("All checks passed")